In [ ]:
#!/usr/bin/env python3
"""
Simple inference script that works with your gymnasium version
"""
import gymnasium as gym
from stable_baselines3 import PPO
import numpy as np
import os
from datetime import datetime

print("Loading PPO model...")
model = PPO.load("./expert_implementations/logs/ppo/CarRacing-v3_6/best_model.zip")

# Create environment
print("Creating CarRacing-v3 environment...")
env = gym.make("CarRacing-v3", continuous=True, render_mode="rgb_array")

# Apply basic preprocessing
from gymnasium.wrappers import ResizeObservation, GrayscaleObservation

env = ResizeObservation(env, shape=(64, 64))
env = GrayscaleObservation(env, keep_dim=False)

# Manual frame stacking implementation
class SimpleFrameStack(gym.Wrapper):
    def __init__(self, env, n_frames=2):
        super().__init__(env)
        self.n_frames = n_frames
        self.frames = []
        
    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        # Initialize with the same frame repeated
        self.frames = [obs.copy() for _ in range(self.n_frames)]
        return np.stack(self.frames, axis=0), info
        
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        # Add new frame and remove oldest
        self.frames.append(obs.copy())
        if len(self.frames) > self.n_frames:
            self.frames.pop(0)
        stacked_obs = np.stack(self.frames, axis=0)
        return stacked_obs, reward, terminated, truncated, info

# Apply frame stacking
env = SimpleFrameStack(env, n_frames=2)

print(f"Environment observation space: {env.observation_space}")
print(f"Environment action space: {env.action_space}")

# Wrap with video recording
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
video_folder = f"videos_{timestamp}"
os.makedirs(video_folder, exist_ok=True)

env = gym.wrappers.RecordVideo(
    env, 
    video_folder=video_folder,
    episode_trigger=lambda x: True,
    name_prefix="ppo_carracing"
)

print(f"Video will be saved to: {video_folder}")
print("Starting inference...")
print("Press Ctrl+C to stop")

try:
    total_rewards = []
    episode_data = []
    
    for episode in range(3):
        print(f"\n=== Episode {episode + 1} ===")
        obs, info = env.reset()
        print(f"Observation shape: {obs.shape}")
        
        episode_reward = 0
        step_count = 0
        max_steps = 1000
        
        while step_count < max_steps:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            episode_reward += reward
            step_count += 1
            
            if step_count % 100 == 0:
                print(f"Episode {episode+1}, Step {step_count}, Reward: {episode_reward:.2f}")
            
            if done:
                print(f"Episode ended: terminated={terminated}, truncated={truncated}")
                break
        
        total_rewards.append(episode_reward)
        episode_data.append({
            'episode': episode + 1,
            'reward': episode_reward,
            'steps': step_count
        })
        print(f"Episode {episode+1} completed. Reward: {episode_reward:.2f}, Steps: {step_count}")
    
    print(f"\n=== Final Results ===")
    print(f"Average reward: {np.mean(total_rewards):.2f}")
    print(f"Reward std: {np.std(total_rewards):.2f}")
    print(f"Min reward: {np.min(total_rewards):.2f}")
    print(f"Max reward: {np.max(total_rewards):.2f}")
    
    # Save results
    results_file = f"results_{timestamp}.txt"
    with open(results_file, 'w') as f:
        f.write("=== PPO CarRacing-v3 Results ===\n")
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Average Reward: {np.mean(total_rewards):.2f}\n")
        f.write(f"Reward Std: {np.std(total_rewards):.2f}\n")
        f.write(f"Min Reward: {np.min(total_rewards):.2f}\n")
        f.write(f"Max Reward: {np.max(total_rewards):.2f}\n")
        f.write("\nEpisode Details:\n")
        for data in episode_data:
            f.write(f"Episode {data['episode']}: Reward={data['reward']:.2f}, Steps={data['steps']}\n")
    
    print(f"Results saved to: {results_file}")
    print(f"Videos saved to: {video_folder}")
    
except KeyboardInterrupt:
    print("\nStopping inference...")
except Exception as e:
    print(f"Error: {e}")
finally:
    env.close()


Loading PPO model...
Creating CarRacing-v3 environment...
Environment observation space: Box(0, 255, (64, 64), uint8)
Environment action space: Box([-1.  0.  0.], 1.0, (3,), float32)
Video will be saved to: videos_20251022_012428
Starting inference...
Press Ctrl+C to stop

=== Episode 1 ===
Observation shape: (2, 64, 64)
Episode 1, Step 100, Reward: 40.54
Episode 1, Step 200, Reward: 113.57
Episode 1, Step 300, Reward: 204.66
Episode 1, Step 400, Reward: 310.18
Episode 1, Step 500, Reward: 437.36
Episode 1, Step 600, Reward: 571.77
Episode 1, Step 700, Reward: 713.39
Episode 1, Step 800, Reward: 779.21
Episode 1, Step 900, Reward: 769.21
Episode 1, Step 1000, Reward: 759.21
Episode ended: terminated=False, truncated=True
Episode 1 completed. Reward: 759.21, Steps: 1000

=== Episode 2 ===
Observation shape: (2, 64, 64)
Episode 2, Step 100, Reward: 28.15
Episode 2, Step 200, Reward: 80.82
Episode 2, Step 300, Reward: 144.39
Episode 2, Step 400, Reward: 224.31
Episode 2, Step 500, Reward: